# mmWave Person Identification

This notebook presents a mmWave radar person-identification pipeline using two classical machine-learning algorithms: **Logistic Regression** and **Random Forest**.

**Pipeline:** data loading → 5-frame stacking → exploration → feature extraction → two-model comparison → leakage-aware evaluation → results.

The raw MiliPoint dataset is kept outside GitHub because of its size. Put it in `data/raw/` before running the notebook.


## 1. Environment

Run this cell first to verify Python and the core packages used by the notebook.


In [1]:
from pathlib import Path
import sys, os, json, pickle, warnings
import numpy as np
import sklearn
import matplotlib
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()

# If the notebook opened inside a subfolder, move up to the project folder.
if not (PROJECT_ROOT / "raw").exists() and (PROJECT_ROOT.parent / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Project root:", PROJECT_ROOT.resolve())
print("Dataset folder:", (PROJECT_ROOT / "raw").resolve())
print("Found id.json:", (PROJECT_ROOT / "raw" / "id.json").exists())

Python: 3.10.20
NumPy: 2.2.6
scikit-learn: 1.7.2
Matplotlib: 3.10.9
Project root: /Users/shabab/Downloads/shajid project revised
Dataset folder: /Users/shabab/Downloads/shajid project revised/raw
Found id.json: True


## 2. Pipeline implementation

The following cells contain the functions from `mmwave_identification.py`, grouped by purpose.


In [2]:
STACKS = 5           # s : consecutive frames stacked into one sample
MAX_POINTS = 22      # k : fixed points per frame (sub-sample / zero-pad)
N_CLASSES = 11       # MiliPoint participants
SPLIT_SEED = 20      # matches the project config
FPS = 24

RAW = "/Users/shabab/Downloads/shajid project revised/raw"
OUT_RES = "results/model_comparison"
OUT_FIG = "results/figures"
PEOPLE = [f"P{i}" for i in range(N_CLASSES)]




### Data loading

Loads radar sessions, keeps a fixed number of points per frame, and creates 5-frame samples.


In [3]:
def load_stacked(frames_per_session=2000, stacks=STACKS, k=MAX_POINTS, seed=SPLIT_SEED):
    """
    Load raw radar frames and build stacked samples, reproducing the official
    MiliPoint loader exactly.

    For every frame: if it has more than k points, randomly sub-sample to k;
    if fewer, zero-pad up to k. Then concatenate `stacks` consecutive frames
    ALONG THE POINT AXIS -> (stacks*k, 3).

    NOTE the flattening. The official loader does NOT keep a separate time
    dimension, which is why permutation-invariant models cannot tell which
    frame a point came from. Restoring that structure is contribution #2.

    Returns
        X    (N, stacks*k, 3) float32
        y    (N,) int         participant id
        sess (N,) int         source session (for the leak-free split)
        pos  (N,) float       relative position within the session, 0..1
    """
    idpath = os.path.join(RAW, "id.json")
    if not os.path.exists(idpath):
        sys.exit(f"ERROR: {idpath} not found. Put the dataset in {RAW}/ "
                 "(see ARTEFACT_README.md).")
    idmap = json.load(open(idpath))
    rng = np.random.default_rng(seed)
    X, y, sess_id, pos = [], [], [], []

    for pid in sorted(idmap, key=int):
        for sess in idmap[pid]:
            path = os.path.join(RAW, f"{sess}.pkl")
            if not os.path.exists(path):
                sys.exit(f"ERROR: missing {path}")
            data = pickle.load(open(path, "rb"))
            if frames_per_session:
                data = data[:frames_per_session]

            fixed = []
            for fr in data:
                p = np.asarray(fr["x"], dtype=np.float32)
                if p.shape[0] >= k:
                    p = p[rng.choice(p.shape[0], k, replace=False)]
                else:
                    p = np.vstack([p, np.zeros((k - p.shape[0], 3), np.float32)])
                fixed.append(p)

            n_win = len(fixed) - stacks + 1
            for i in range(n_win):
                X.append(np.vstack(fixed[i:i + stacks]))
                y.append(int(pid))
                sess_id.append(sess)
                pos.append(i / max(n_win - 1, 1))

    return (np.asarray(X, dtype=np.float32), np.asarray(y),
            np.asarray(sess_id), np.asarray(pos, dtype=np.float32))




### Evaluation protocols

Defines the random split and the block split used to test the effect of overlapping windows.


In [4]:
def split_random(n, seed=SPLIT_SEED, tr=0.8, va=0.1):
    """Published protocol: shuffle every window, then cut. LEAKY (see header)."""
    idx = np.random.default_rng(seed).permutation(n)
    a, b = int(tr * n), int((tr + va) * n)
    return idx[:a], idx[a:b], idx[b:]


def split_block(pos, sess, tr=0.8, va=0.1, guard=0.01):
    """
    Leak-free: contiguous time blocks WITHIN each session, plus guard bands so
    no training window shares a frame with a test window.

    A session-wise split is not usable here: several participants (P3, P5, P6,
    P7, P9, P10) were recorded in only ONE session, so holding out whole
    sessions would delete those classes entirely. Blocking within sessions
    keeps all 11 participants in every partition.
    """
    tr_i, va_i, te_i = [], [], []
    for s_ in np.unique(sess):
        m = np.where(sess == s_)[0]
        p = pos[m]
        tr_i.append(m[p < tr - guard])
        va_i.append(m[(p > tr + guard) & (p < tr + va - guard)])
        te_i.append(m[p > tr + va + guard])
    return np.concatenate(tr_i), np.concatenate(va_i), np.concatenate(te_i)




### Metrics

Defines the classification metrics used in the experiments.


In [5]:
def confusion_matrix(y_true, y_pred, n_cls=N_CLASSES):
    cm = np.zeros((n_cls, n_cls), dtype=np.int64)
    for t, p in zip(np.asarray(y_true).astype(int).ravel(),
                    np.asarray(y_pred).astype(int).ravel()):
        cm[t, p] += 1
    return cm


def per_class_report(cm):
    """
    Per-class precision / recall / F1 plus balanced accuracy and macro-F1.

    The published benchmark reports ONLY aggregate top-1 accuracy. Because the
    classes are imbalanced by 3.67x, that number is dominated by the
    well-represented participants and can hide near-total failure on the rare
    ones. This is contribution #1.
    """
    tp = np.diag(cm).astype(float)
    support = cm.sum(1).astype(float)
    predicted = cm.sum(0).astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        recall = np.where(support > 0, tp / support, np.nan)
        precision = np.where(predicted > 0, tp / predicted, np.nan)
        f1 = np.where((precision + recall) > 0,
                      2 * precision * recall / (precision + recall), 0.0)
    total = cm.sum()
    return {
        "confusion_matrix": cm, "support": support,
        "precision": precision, "recall": recall, "f1": f1,
        "accuracy": float(tp.sum() / total) if total else np.nan,
        "balanced_accuracy": float(np.nanmean(recall)),
        "macro_f1": float(np.nanmean(f1)),
        "weighted_f1": float(np.nansum(f1 * support) / support.sum()) if support.sum() else np.nan,
    }


def evaluate(y_true, y_pred, proba=None, n_cls=N_CLASSES):
    rep = per_class_report(confusion_matrix(y_true, y_pred, n_cls))
    if proba is not None and np.ndim(proba) == 2 and proba.shape[1] >= 3:
        top3 = np.argsort(-proba, axis=1)[:, :3]
        rep["top3_accuracy"] = float((top3 == np.asarray(y_true)[:, None]).any(1).mean())
    else:
        rep["top3_accuracy"] = np.nan
    return rep


def format_report(rep, names=None):
    n = rep["confusion_matrix"].shape[0]
    names = names or PEOPLE
    L = ["Per-class performance", "-" * 62,
         f"{'person':>8} {'support':>9} {'precision':>10} {'recall':>8} {'f1':>8}", "-" * 62]
    for i in range(n):
        L.append(f"{names[i]:>8} {int(rep['support'][i]):>9} "
                 f"{rep['precision'][i]:>10.3f} {rep['recall'][i]:>8.3f} {rep['f1'][i]:>8.3f}")
    L += ["-" * 62,
          f"{'top-1 accuracy':<26}{rep['accuracy']:.4f}",
          f"{'top-3 accuracy':<26}{rep['top3_accuracy']:.4f}",
          f"{'balanced accuracy':<26}{rep['balanced_accuracy']:.4f}",
          f"{'macro-F1':<26}{rep['macro_f1']:.4f}", ""]
    gap = rep["accuracy"] - rep["balanced_accuracy"]
    L.append(f"accuracy - balanced accuracy = {gap:+.4f}  "
             "(positive => weaker on under-represented participants)")
    return "\n".join(L)


def top_confusions(cm, k=5, names=None):
    names = names or PEOPLE
    n = cm.shape[0]
    pairs = [(cm[i, j], names[i], names[j])
             for i in range(n) for j in range(n) if i != j and cm[i, j] > 0]
    pairs.sort(reverse=True)
    return [{"count": int(c), "true": t, "pred": p} for c, t, p in pairs[:k]]




### Feature engineering

Builds fixed-length geometric, sparsity and motion-related features for selected models.


In [6]:
_PER_FRAME = ["n_points", "cx", "cy", "cz", "sx", "sy", "sz",
              "rx", "ry", "rz", "r_mean", "r_std", "bbox_vol", "spread"]


def _frame_stats(pts):
    if pts.shape[0] == 0:                      # frames with ZERO detections exist
        return np.zeros(len(_PER_FRAME))
    c, s = pts.mean(0), pts.std(0)
    rng = pts.max(0) - pts.min(0)
    r = np.linalg.norm(pts, axis=1)
    return np.array([pts.shape[0], *c, *s, *rng, r.mean(), r.std(),
                     float(np.prod(np.maximum(rng, 1e-6))),
                     np.linalg.norm(pts - c, axis=1).mean()])


def _feature_names():
    names = [f"{f}_{a}" for f in _PER_FRAME for a in ("mean", "std", "min", "max")]
    names += ["disp_mean", "disp_std", "disp_max", "disp_total",
              "vx_mean", "vy_mean", "vz_mean", "vx_std", "vy_std", "vz_std",
              "accel_mean", "accel_std"]
    names += ["g_cx", "g_cy", "g_cz", "g_sx", "g_sy", "g_sz",
              "g_rx", "g_ry", "g_rz", "g_r_mean", "g_r_std",
              "g_spread", "g_n_total", "g_n_frames_empty"]
    return names


FEATURE_NAMES = _feature_names()


def feature_groups():
    g = {}
    for nm in FEATURE_NAMES:
        if nm.startswith(("disp", "vx", "vy", "vz", "accel")):
            g[nm] = "motion / gait"
        elif nm.startswith("n_points") or nm.startswith("g_n_"):
            g[nm] = "sparsity"
        elif nm.startswith("g_"):
            g[nm] = "global geometry"
        else:
            g[nm] = "per-frame geometry"
    return g


def extract_features(samples, stacks=STACKS, k=MAX_POINTS, verbose=True):
    """(N, stacks*k, 3) point clouds -> (N, 82) feature matrix."""
    samples = np.asarray(samples, dtype=np.float64)
    n = samples.shape[0]
    out = np.zeros((n, len(FEATURE_NAMES)))

    for i in range(n):
        if verbose and n > 5000 and i % 10000 == 0 and i:
            print(f"    {i:,}/{n:,}", end="\r")
        frames = samples[i].reshape(stacks, k, 3)

        pf, cen, empties = [], [], 0
        for f in range(stacks):
            real = frames[f][np.abs(frames[f]).sum(1) > 0]     # drop padding
            if real.shape[0] == 0:
                empties += 1
            pf.append(_frame_stats(real))
            cen.append(real.mean(0) if real.shape[0] else np.zeros(3))
        pf = np.stack(pf); cen = np.stack(cen)

        agg = np.stack([pf.mean(0), pf.std(0), pf.min(0), pf.max(0)], axis=1).ravel()

        d = np.diff(cen, axis=0)
        dist = np.linalg.norm(d, axis=1)
        accel = np.diff(dist) if dist.size > 1 else np.zeros(1)
        motion = np.array([dist.mean(), dist.std(), dist.max(), dist.sum(),
                           *d.mean(0), *d.std(0), accel.mean(), accel.std()])

        allp = samples[i]
        real_all = allp[np.abs(allp).sum(1) > 0]
        if real_all.shape[0]:
            gc = real_all.mean(0)
            gr = np.linalg.norm(real_all, axis=1)
            glob = np.array([*gc, *real_all.std(0),
                             *(real_all.max(0) - real_all.min(0)),
                             gr.mean(), gr.std(),
                             np.linalg.norm(real_all - gc, axis=1).mean(),
                             real_all.shape[0], empties])
        else:
            glob = np.zeros(14)

        out[i] = np.concatenate([agg, motion, glob])

    if verbose and n > 5000:
        print(" " * 30, end="\r")
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)




### Selected models

The comparison focuses on two complementary classifiers: Logistic Regression provides a linear, interpretable baseline, while Random Forest can learn nonlinear feature interactions. Both use the same engineered feature matrix and evaluation splits.


In [7]:
def build_models(seed=0):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    return {
        "Logistic Regression": make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, random_state=seed)
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=300, n_jobs=-1, random_state=seed
        ),
    }


### Figures

Defines the dataset and results visualisations.


In [8]:
def _plt():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 9,
                         "axes.spines.top": False, "axes.spines.right": False,
                         "axes.grid": True, "grid.alpha": .25})
    return plt


C_MAIN, C_ALT, C_WARN, C_MUT = "#2E5395", "#7BAFD4", "#C44E52", "#8C8C8C"


def _save(fig, name):
    os.makedirs(OUT_FIG, exist_ok=True)
    for ext in ("png", "pdf"):
        fig.savefig(f"{OUT_FIG}/{name}.{ext}", bbox_inches="tight")
    _plt().close(fig)
    print(f"    {OUT_FIG}/{name}.png")


def figures_dataset(X, y, sess):
    """Sparsity, class balance and example point clouds."""
    plt = _plt()
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

    frames = X.reshape(-1, STACKS, MAX_POINTS, 3)
    counts = (np.abs(frames).sum(-1) > 0).sum(-1).ravel()

    fig, ax = plt.subplots(1, 2, figsize=(9, 3.1))
    ax[0].hist(counts, bins=np.arange(0, counts.max() + 2) - .5, color=C_MAIN,
               edgecolor="white", linewidth=.4)
    ax[0].axvline(MAX_POINTS, color=C_WARN, ls="--", lw=1.6, label=f"max_points = {MAX_POINTS}")
    ax[0].set_xlabel("detected points per frame"); ax[0].set_ylabel("frames")
    ax[0].set_title("Point-count distribution"); ax[0].legend(frameon=False, fontsize=8)
    ax[0].annotate(f"median {int(np.median(counts))}   max {counts.max()}",
                   xy=(.97, .8), xycoords="axes fraction", ha="right", fontsize=8, color=C_MUT)
    pad = np.clip(1 - counts / MAX_POINTS, 0, 1) * 100
    ax[1].hist(pad, bins=30, color=C_ALT, edgecolor="white", linewidth=.4)
    ax[1].axvline(pad.mean(), color=C_WARN, ls="--", lw=1.6, label=f"mean {pad.mean():.0f}% padding")
    ax[1].set_xlabel("share of the input tensor that is padding (%)")
    ax[1].set_ylabel("frames"); ax[1].set_title("Zero-padding per frame")
    ax[1].legend(frameon=False, fontsize=8)
    fig.suptitle("Radar sparsity: sub-sampling never fires; every frame is padded", y=1.03)
    _save(fig, "fig01_sparsity")

    ids, cnt = np.unique(y, return_counts=True)
    fig, ax = plt.subplots(figsize=(6.4, 2.9))
    bars = ax.bar([f"P{i}" for i in ids], cnt, color=C_MAIN, edgecolor="white", linewidth=.5)
    for b in bars:
        if b.get_height() == cnt.max(): b.set_color(C_ALT)
        if b.get_height() == cnt.min(): b.set_color(C_WARN)
    ax.set_xlabel("participant"); ax.set_ylabel("samples")
    ax.set_title(f"Class imbalance - ratio {cnt.max()/cnt.min():.2f}×")
    _save(fig, "fig02_class_balance")

    fig = plt.figure(figsize=(10, 2.8))
    for j, pid in enumerate(ids[:4]):
        s = X[np.where(y == pid)[0][0]]
        p = s[np.abs(s).sum(1) > 0]
        a = fig.add_subplot(1, 4, j + 1, projection="3d")
        a.scatter(p[:, 0], p[:, 1], p[:, 2], s=22, c=p[:, 2], cmap="viridis", depthshade=False)
        a.set_title(f"P{pid} - {len(p)} points", fontsize=8)
        a.tick_params(labelsize=5); a.grid(False)
    fig.suptitle("A person, as the radar sees them (5 stacked frames)", y=1.04)
    _save(fig, "fig03_point_clouds")


def figures_results(rows, preds):
    plt = _plt()
    protos = sorted({r["protocol"] for r in rows}, reverse=True)
    models = []
    for r in rows:
        if r["model"] not in models:
            models.append(r["model"])

    def g(m, p, k="top1"):
        v = [r[k] for r in rows if r["model"] == m and r["protocol"] == p]
        return float(np.mean(v)) if v else np.nan

    def gs(m, p, k="top1"):
        v = [r[k] for r in rows if r["model"] == m and r["protocol"] == p]
        return float(np.std(v, ddof=1)) if len(v) > 1 else 0.0

    models.sort(key=lambda m: -g(m, protos[0]))

    yy = np.arange(len(models)); h = .38
    fig, ax = plt.subplots(figsize=(7.6, .46 * len(models) + 1.7))
    if len(protos) > 1:
        ax.barh(yy + h/2, [g(m, "random")*100 for m in models], h, color=C_ALT,
                xerr=[gs(m, "random")*100 for m in models], error_kw=dict(ecolor=C_MUT, lw=.8),
                label="random split (published protocol)")
        ax.barh(yy - h/2, [g(m, "block")*100 for m in models], h, color=C_MAIN,
                xerr=[gs(m, "block")*100 for m in models], error_kw=dict(ecolor=C_MUT, lw=.8),
                label="block split (leak-free)")
    else:
        ax.barh(yy, [g(m, protos[0])*100 for m in models], h*2, color=C_MAIN)
    ax.axvline(100/N_CLASSES, color=C_WARN, ls=":", lw=1.4, label="chance (9.1%)")
    ax.set_yticks(yy); ax.set_yticklabels(models); ax.invert_yaxis()
    ax.set_xlabel("top-1 accuracy (%)"); ax.set_xlim(0, 100)
    ax.set_title("Identification accuracy - selected models")
    ax.legend(frameon=False, fontsize=8, loc="lower right")
    _save(fig, "fig04_model_comparison")

    if len(protos) > 1:
        ms = [m for m in models if m != "Majority baseline"]
        rnd = [g(m, "random")*100 for m in ms]; blk = [g(m, "block")*100 for m in ms]
        fig, ax = plt.subplots(figsize=(7.4, 3.8))
        for i, (r_, b_) in enumerate(zip(rnd, blk)):
            ax.plot([0, 1], [r_, b_], "-o", color=C_MAIN, ms=5, lw=1.3)
            ax.annotate(ms[i], xy=(1.03, b_), fontsize=7.5, va="center", color=C_MUT)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["random split\n(overlapping windows)", "block split\n(leak-free)"])
        ax.set_xlim(-.15, 1.6); ax.set_ylabel("top-1 accuracy (%)")
        ax.axhline(100/N_CLASSES, color=C_WARN, ls=":", lw=1.2)
        ax.set_title(f"Evaluation-protocol leakage: mean drop "
                     f"{np.mean(np.array(rnd)-np.array(blk)):.1f} points")
        _save(fig, "fig05_leakage")

    for proto in protos:
        keys = [k for k in preds if k.startswith(proto + "|")]
        if not keys:
            continue
        bk = max(keys, key=lambda k: preds[k]["rep"]["accuracy"])
        rep = preds[bk]["rep"]; name = bk.split("|", 1)[1]
        cm = rep["confusion_matrix"].astype(float)
        cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)

        fig, ax = plt.subplots(figsize=(5.6, 4.8))
        im = ax.imshow(cmn, cmap="viridis", vmin=0, vmax=1)
        fig.colorbar(im, ax=ax, fraction=.046, label="proportion of true class")
        ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(PEOPLE, rotation=45, ha="right")
        ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(PEOPLE)
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.grid(False)
        ax.set_title(f"Confusion matrix - {name}\n({proto} split, acc {rep['accuracy']*100:.1f}%)",
                     fontsize=9)
        for i in range(N_CLASSES):
            for j in range(N_CLASSES):
                if cmn[i, j] > .01:
                    ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=6,
                            color="white" if cmn[i, j] < .5 else "black")
        _save(fig, f"fig06_confusion_{proto}")

        rec, sup = rep["recall"], rep["support"]
        order = np.argsort(sup)
        fig, ax = plt.subplots(figsize=(6.6, 2.9))
        ax.bar(range(N_CLASSES), rec[order]*100, color=C_MAIN, edgecolor="white", linewidth=.5)
        ax.axhline(np.nanmean(rec)*100, color=C_WARN, ls="--", lw=1.3,
                   label=f"balanced accuracy {np.nanmean(rec)*100:.1f}%")
        ax.set_xticks(range(N_CLASSES))
        ax.set_xticklabels([f"{PEOPLE[i]}\n{int(sup[i])}" for i in order], fontsize=7)
        ax.set_xlabel("participant (ordered by test support, rarest first)")
        ax.set_ylabel("recall (%)"); ax.set_ylim(0, 100)
        ax.set_title(f"Per-class recall - {name} ({proto} split)")
        ax.legend(frameon=False, fontsize=8)
        _save(fig, f"fig07_per_class_recall_{proto}")

    imp = next((preds[k]["importances"] for k in preds if "importances" in preds[k]), None)
    if imp is not None:
        idx = np.argsort(imp)[::-1][:18]
        fig, ax = plt.subplots(figsize=(6.4, 4.2))
        ax.barh(range(len(idx)), imp[idx][::-1]*100, color=C_MAIN)
        ax.set_yticks(range(len(idx)))
        ax.set_yticklabels([FEATURE_NAMES[i] for i in idx][::-1], fontsize=7)
        ax.set_xlabel("importance (%)"); ax.set_title("Most informative handcrafted features")
        _save(fig, "fig08_feature_importance")

        grp = feature_groups(); agg = {}
        for i, nm in enumerate(FEATURE_NAMES):
            agg[grp[nm]] = agg.get(grp[nm], 0) + imp[i]
        ks = sorted(agg, key=agg.get, reverse=True)
        fig, ax = plt.subplots(figsize=(5.4, 2.8))
        ax.bar(ks, [agg[k]*100 for k in ks], color=[C_MAIN, C_ALT, C_MUT, C_WARN][:len(ks)],
               edgecolor="white")
        ax.set_ylabel("total importance (%)")
        ax.set_title("Which kind of information carries identity?")
        plt.setp(ax.get_xticklabels(), rotation=15, ha="right", fontsize=8)
        _save(fig, "fig09_feature_groups")




## 3. Experiment settings

Use a small frame count for a live demonstration. Increase the frame count and number of seeds for the final dissertation run.

`split="both"` is recommended because it shows the difference between random and leakage-resistant evaluation.


In [9]:
from types import SimpleNamespace

CONFIG = SimpleNamespace(
    frames=500,          # 200/500 for demo; 2000 for the normal run; 0 = all available frames
    split="both",        # compare random and leakage-resistant block evaluation
    seeds=[0],            # final study can use [0, 1, 2]
    class_weighted=False,
    explore=True,
)

print(CONFIG)


namespace(frames=500, split='both', seeds=[0], class_weighted=False, explore=True)


## 4. Load the dataset

This is the first experiment step. Check the number of samples, participants and sessions before continuing.


In [12]:
import time
t0 = time.time()
X, y, sess, pos = load_stacked(
    frames_per_session=CONFIG.frames,
    stacks=STACKS,
    k=MAX_POINTS,
    seed=SPLIT_SEED,
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("participants:", np.unique(y))
print("sessions:", len(np.unique(sess)))
print("load time:", f"{time.time()-t0:.1f}s")


X shape: (9424, 110, 3)
y shape: (9424,)
participants: [ 0  1  2  3  4  5  6  7  8  9 10]
sessions: 19
load time: 1.1s


## 5. Explore the data

These figures provide a visual check of the radar point clouds, sparsity and class distribution.


In [ ]:
figures_dataset(X, y, sess)
print("Figures written to:", OUT_FIG)


## 6. Extract engineered features

The handcrafted features provide the input for the selected models.


In [13]:
import time
t0 = time.time()
F = extract_features(X)
print("Feature matrix:", F.shape)
print("Feature extraction time:", f"{time.time()-t0:.1f}s")


Feature matrix: (9424, 82)    
Feature extraction time: 2.1s


## 7. Compare evaluation protocols


In [14]:
protocol_indices = {}
for proto in ("random", "block"):
    if proto == "random":
        tr, va, te = split_random(len(X))
    else:
        tr, va, te = split_block(pos, sess)
    protocol_indices[proto] = (tr, va, te)
    print(f"{proto:>6}: train={len(tr):,}  val={len(va):,}  test={len(te):,}")


random: train=7,539  val=942  test=943
 block: train=7,448  val=760  test=855


## 8. Logistic Regression and Random Forest benchmark

Compare both selected models on the same feature set and evaluation splits.


In [15]:
classical_rows = []
classical_preds = {}

for proto, (tr, va, te) in protocol_indices.items():
    print("\n" + "="*72)
    print("PROTOCOL:", proto.upper())
    print("="*72)
    for name, model in build_models(CONFIG.seeds[0]).items():
        t0 = time.time()
        model.fit(F[tr], y[tr])
        yp = model.predict(F[te])
        proba = model.predict_proba(F[te]) if hasattr(model, "predict_proba") else None
        rep = evaluate(y[te], yp, proba)
        classical_rows.append({
            "model": name, "protocol": proto, "seed": CONFIG.seeds[0],
            "top1": rep["accuracy"], "top3": rep["top3_accuracy"],
            "balanced": rep["balanced_accuracy"], "macro_f1": rep["macro_f1"],
            "seconds": time.time()-t0
        })
        classical_preds[f"{proto}|{name}"] = {"rep": rep}
        print(f"{name:<32} top1={rep['accuracy']*100:6.2f}%  "
              f"balanced={rep['balanced_accuracy']*100:6.2f}%  "
              f"macro-F1={rep['macro_f1']*100:6.2f}%")



PROTOCOL: RANDOM
Logistic Regression              top1= 66.81%  balanced= 70.73%  macro-F1= 69.95%
Random Forest                    top1= 89.61%  balanced= 88.62%  macro-F1= 89.55%

PROTOCOL: BLOCK
Logistic Regression              top1= 59.06%  balanced= 59.63%  macro-F1= 60.02%
Random Forest                    top1= 61.87%  balanced= 54.34%  macro-F1= 57.30%


## 9. Save and inspect results

The table summarises both selected models across the configured evaluation protocols and saves a CSV under `results/model_comparison/`.


In [16]:
import pandas as pd

all_rows = classical_rows
results_df = pd.DataFrame(all_rows)

if len(results_df):
    display(results_df.sort_values(["protocol", "top1"], ascending=[True, False]))
    out_dir = PROJECT_ROOT / "results" / "model_comparison"
    out_dir.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(out_dir / "notebook_results.csv", index=False)
    print("Saved:", out_dir / "notebook_results.csv")
else:
    print("No results yet.")


,model,protocol,seed,top1,top3,balanced,macro_f1,seconds
3,Random Forest,block,0,0.618713,0.842105,0.543434,0.572953,1.581771
2,Logistic Regression,block,0,0.590643,0.880702,0.596296,0.600202,0.750810
1,Random Forest,random,0,0.896076,0.985154,0.886222,0.895453,1.771501
0,Logistic Regression,random,0,0.668081,0.920467,0.707327,0.699550,0.652880


Saved: /Users/shabab/Downloads/shajid project revised/results/model_comparison/notebook_results.csv


## 10. Final figures

Generate the comparison figures after the two models have completed.


In [17]:
combined_preds = classical_preds

if len(all_rows) and combined_preds:
    figures_results(all_rows, combined_preds)
    best_key = max(combined_preds, key=lambda k: combined_preds[k]["rep"]["accuracy"])
    best_rep = combined_preds[best_key]["rep"]
    print("\nBest model:", best_key)
    print(format_report(best_rep))
else:
    print("Run the model comparison before generating final results.")


    results/figures/fig04_model_comparison.png
    results/figures/fig05_leakage.png
    results/figures/fig06_confusion_random.png
    results/figures/fig07_per_class_recall_random.png
    results/figures/fig06_confusion_block.png
    results/figures/fig07_per_class_recall_block.png

Best model: random|Random Forest
Per-class performance
--------------------------------------------------------------
  person   support  precision   recall       f1
--------------------------------------------------------------
      P0       166      0.861    0.934    0.896
      P1       198      0.902    0.929    0.915
      P2        82      0.835    0.927    0.879
      P3        61      0.967    0.951    0.959
      P4       102      0.841    0.882    0.861
      P5        52      0.979    0.904    0.940
      P6        49      0.865    0.918    0.891
      P7        46      1.000    0.804    0.892
      P8       101      0.964    0.792    0.870
      P9        45      0.895    0.756    0.819
     

## 11. Supervisor presentation order

1. Problem and motivation.
2. Dataset and 5-frame preprocessing.
3. Why Logistic Regression and Random Forest were selected.
4. Random versus block evaluation.
5. Why overlapping windows can inflate performance.
6. Accuracy, balanced accuracy and macro-F1.
7. Model comparison results and reproducibility.

**Key explanation:** consecutive radar windows overlap. A random split can therefore put very similar observations in both training and testing. The block split with guard bands provides a more realistic generalisation estimate.
